In [5]:
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

from torch.optim import AdamW
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)
import random

from pathlib import Path
import json
sys.path.append('../')
from data_provider.data_loader import load_dataset


#### Experiment Configuration

In [ ]:
# Data and Model Configurations
DATA_FOLDER = '../data'
DATASET_NAME = 'experiment3'

# Choose a pre-trained BERT model
MODEL_NAME = 'bert-base-uncased'

# Data Preprocessing
TEST_SIZE = 0.2
VAL_SIZE = 0.1 
RANDOM_SEED = 2026
MAX_LENGTH = 256

# Model Hyperparameters
WEIGHT_DECAY = 0.01

# Trainin and Optimization
BATCH_SIZE = 16
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1

# Early Stopping
PATIENCE = 2
MIN_DELTA = 0.001

# Reproducibility
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# Device Configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE


device(type='cpu')

#### Load and Split Dataset

In [ ]:
df = load_dataset(DATA_FOLDER, DATASET_NAME)

# Check the column names to ensure features and labels are loaded correctly
if isinstance(df, dict):
    print(df['test'].head())
else:
    print(df.head())

                                                text  label
0  Phones\n\nModern humans today are always on th...      0
1  This essay will explain if drivers should or s...      0
2  Driving while the use of cellular devices\n\nT...      0
3  Phones & Driving\n\nDrivers should not be able...      0
4  Cell Phone Operation While Driving\n\nThe abil...      0


#### Split Dataset

In [ ]:
# Dataset is not randomly ordered, so we will shuffle it to ensure a representative split between training and testing sets

if isinstance(df, dict):
    train_val_df = df['train']
    test_df = df['test']
else:
    train_val_df, test_df = train_test_split(
        df,
        test_size=TEST_SIZE,
        random_state=RANDOM_SEED,
        stratify=df['label'],
        shuffle=True
    )

train_df, val_df = train_test_split(
    train_val_df,
    test_size=(VAL_SIZE / (1.0 - TEST_SIZE)),
    random_state=RANDOM_SEED,
    stratify=train_val_df['label'],
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f'Train set size: {len(train_df)}')
print(f'Validation set size: {len(val_df)}')
print(f'Test set size: {len(test_df)}')


Train set size: 31407
Validation set size: 4487
Test set size: 8974


#### Tokenizer and DataLoaders

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TransformerDataset(Dataset):
    def __init__(self, df, tokenizer, max_length):
        self.texts = df['text'].tolist()
        self.labels = df['label'].astype(int).tolist()
        self.encodings = tokenizer(self.texts, truncation=True, padding='max_length', max_length=max_length)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            'input_ids': torch.tensor(self.encodings['input_ids'][idx], dtype=torch.long),
            'attention_mask': torch.tensor(self.encodings['attention_mask'][idx], dtype=torch.long),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long),
        }
        
        return item


train_dataset = TransformerDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = TransformerDataset(val_df, tokenizer, MAX_LENGTH)
test_dataset = TransformerDataset(test_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


#### Model Setup

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_training_steps = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_training_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_training_steps,)

model


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

#### Training and Evaluation

In [ ]:
def run_epoch(model, data_loader, optimizer=None, scheduler=None):
    is_training = optimizer is not None
    model.train() if is_training else model.eval()

    total_loss = 0.0
    y_true = []
    y_pred = []
    y_scores = []

    def process_batch(batch):
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        logits = outputs.logits

        if is_training:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if scheduler is not None:
                scheduler.step()

        return loss, logits, labels

    if is_training:
        for batch in tqdm(data_loader):
            loss_val, logits, labels = process_batch(batch)

            total_loss += loss_val.item() * batch['input_ids'].size(0)
            probabilities = torch.softmax(logits, dim=1)[:, 1]
            predictions = torch.argmax(logits, dim=1)

            y_true.extend(labels.detach().cpu().numpy())
            y_pred.extend(predictions.detach().cpu().numpy())
            y_scores.extend(probabilities.detach().cpu().numpy())
    else:
        with torch.no_grad():
            for batch in tqdm(data_loader):
                loss_val, logits, labels = process_batch(batch)

                total_loss += loss_val.item() * batch['input_ids'].size(0)
                probabilities = torch.softmax(logits, dim=1)[:, 1]
                predictions = torch.argmax(logits, dim=1)
                
                y_true.extend(labels.detach().cpu().numpy())
                y_pred.extend(predictions.detach().cpu().numpy())
                y_scores.extend(probabilities.detach().cpu().numpy())

    average_loss = total_loss / len(data_loader.dataset)
    accuracy = accuracy_score(y_true, y_pred)
    
    return average_loss, accuracy, y_true, y_pred, y_scores


best_val_accuracy = 0.0
best_model_state = None
epochs_without_improvement = 0

for epoch in range(NUM_EPOCHS):
    train_loss, train_accuracy, train_true, train_pred, train_scores = run_epoch(model, train_loader, optimizer, scheduler)
    val_loss, val_accuracy, val_true, val_pred, val_scores = run_epoch(model, val_loader)

    print(
        f'Epoch {epoch + 1}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_accuracy:.4f} | '
        f'Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.4f}'
    )

    if val_accuracy > best_val_accuracy + MIN_DELTA:
        best_val_accuracy = val_accuracy
        best_model_state = {
            key: value.detach().cpu().clone() for key, value in model.state_dict().items()
        }
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        break

if best_model_state is not None:
    model.load_state_dict(best_model_state)

test_loss, test_accuracy, y_true, y_pred, y_scores = run_epoch(model, test_loader)
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy:.4f}')
print(classification_report(y_true, y_pred, target_names=['Human', 'AI']))
print(f'Test ROC-AUC: {roc_auc_score(y_true, y_scores):.4f}')

cm = confusion_matrix(y_true, y_pred)
confusion_display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Human', 'AI']).plot(cmap='Blues')
plt.title('BERT Confusion Matrix')
plt.show()

from pathlib import Path
import json

results_dir = Path('../results') / f"bert__{DATASET_NAME}__{MODEL_NAME.replace('/', '_')}"
results_dir.mkdir(parents=True, exist_ok=True)

report_dict = classification_report(y_true, y_pred, target_names=['Human', 'AI'], output_dict=True)
pd.DataFrame(cm, index=['Human', 'AI'], columns=['Predicted Human', 'Predicted AI']).to_csv(results_dir / 'confusion_matrix.csv')
confusion_display.figure_.savefig(results_dir / 'confusion_matrix.png', bbox_inches='tight')

bert_metrics = {
    'dataset_name': DATASET_NAME,
    'model_name': MODEL_NAME,
    'test_loss': float(test_loss),
    'test_accuracy': float(test_accuracy),
    'test_roc_auc': float(roc_auc_score(y_true, y_scores)),
    'batch_size': BATCH_SIZE,
    'num_epochs': NUM_EPOCHS,
    'learning_rate': LEARNING_RATE,
}

with (results_dir / 'metrics.json').open('w', encoding='utf-8') as metrics_file:
    json.dump(bert_metrics, metrics_file, indent=2)

with (results_dir / 'classification_report.json').open('w', encoding='utf-8') as report_file:
    json.dump(report_dict, report_file, indent=2)

if DATASET_NAME == 'experiment3':
    source_results = []

    for source_column, source_subset in test_df.groupby('source'):
        source_texts = source_subset['text'].astype(str).tolist()
        source_encodings = tokenizer(
            source_texts,
            truncation=True,
            padding='max_length',
            max_length=MAX_LENGTH,
            return_tensors='pt',
        )
        source_input_ids = source_encodings['input_ids'].to(DEVICE)
        source_attention_mask = source_encodings['attention_mask'].to(DEVICE)

        with torch.no_grad():
            source_logits = model(input_ids=source_input_ids, attention_mask=source_attention_mask).logits
            source_probabilities = torch.softmax(source_logits, dim=1)[:, 1].cpu().numpy()

        source_predictions = (source_probabilities >= 0.5).astype(int)
        source_results.append({
            'source_model': source_column,
            'samples': len(source_texts),
            'detected_as_ai_rate': float(source_predictions.mean()),
        })

    source_results_df = pd.DataFrame(source_results)
    source_results_df.to_csv(results_dir / 'source_detection_rates.csv', index=False)

    print('Per-source detection rates on multi_model_detection.csv:')
    print(source_results_df)

print(f'Saved BERT results to {results_dir}')


#### Custom Text Prediction

In [12]:
def predict_custom_texts(model, tokenizer, texts, max_length):
    if isinstance(texts, str):
        texts = [texts]

    encodings = tokenizer(
        texts,
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors='pt',
    )

    input_ids = encodings['input_ids'].to(DEVICE)
    attention_mask = encodings['attention_mask'].to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        probabilities = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()

    predictions = (probabilities >= 0.5).astype(int)
    results = pd.DataFrame(
        {
            'text': texts,
            'predicted_label': ['AI' if pred == 1 else 'Human' for pred in predictions],
            'ai_probability': probabilities,
        }
    )
    print(results)
    
    return results


#### Test BERT with Custom Inputs

In [ ]:
custom_texts = [
    'This essay argues that social media can improve civic participation when paired with strong moderation policies.',
    'In conclusion, the provided evidence clearly demonstrates a multifaceted and highly structured perspective on the topic.',
    "This isn't working very well, as it seems to just be predicting AI for longer messages and Human for shorter messages.",
]

predict_custom_texts(model, tokenizer, custom_texts, MAX_LENGTH)


#### Save BERT Artifacts

In [ ]:
from pathlib import Path
import json

SAVE_DIR = Path('../artifacts/bert')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

torch.save(model.state_dict(), SAVE_DIR / 'weights.pt')
tokenizer.save_pretrained(str(SAVE_DIR / 'tokenizer'))

bert_config = {
    'dataset_name': DATASET_NAME,
    'model_name': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'batch_size': BATCH_SIZE,
    'num_epochs': NUM_EPOCHS,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'warmup_ratio': WARMUP_RATIO,
    'patience': PATIENCE,
    'min_delta': MIN_DELTA,
}

with (SAVE_DIR / 'config.json').open('w', encoding='utf-8') as config_file:
    json.dump(bert_config, config_file, indent=2)

print(f'Saved BERT artifacts to {SAVE_DIR}')


#### Reload BERT Artifacts

In [ ]:
from pathlib import Path

RELOAD_DIR = Path('../artifacts/bert')

reloaded_tokenizer = AutoTokenizer.from_pretrained(str(RELOAD_DIR / 'tokenizer'))
reloaded_bert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
reloaded_bert_model.load_state_dict(torch.load(RELOAD_DIR / 'weights.pt', map_location=DEVICE))
reloaded_bert_model.eval()

print('Reloaded BERT artifacts.')
